## About this notebook:

This notebook uses Affinder napari plugin to find transformation manually.

Input:
- pathway to a directory containing segmented labels
- pathway to a directory containing data frames
- pathway to a directory containing transformations (pkl files)
- well and round (as in the names of directories) selected for corrections

Output:
- a set of transforms for the selected well with the corrected transform for a given round

Optionally you can visualize alignment (visualization uses Napari).

## Fill in info about the experiment to process

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']

# Paths derived from experiment config
path_labels = os.path.join(base_dir, 'im_segmented')  # segmented labels
path_df     = os.path.join(base_dir, 'output_df')      # data frames + transforms
path_save   = os.path.join(base_dir, 'output_df')      # save updated transforms here

# Well and round to correct manually — change these as needed
myWell = well_list[0]  # e.g. 'D03'
myRound = 0            # round number to correct (integer)
anchor_round_selected = 1  # anchor round (default 1)

print(f'Base dir: {base_dir}')
print(f'Well list: {well_list}')
print(f'Correcting: well={myWell}, round={myRound}')

## Prepare for processing

In [2]:
import os
import pickle
import pandas as pd
import numpy as np
import napari
from tifffile import imsave,imread
from scipy import ndimage as ndi
from skimage import transform
import matplotlib.pyplot as plt

In [3]:
def read_labels(path_labels,myWell,alignRound,anchorRound):
    
    labels_list = [x for x in os.listdir(os.path.join(path_labels,myWell)) if 'tif' in x]
    labels_list.sort()

    labels_list = [labels_list[int(anchorRound)],labels_list[int(alignRound)]]

    labels_im_list = []
    for lab_im_name in labels_list:

        lab_im = plt.imread(os.path.join(path_labels,myWell,lab_im_name))
        labels_im_list.append(lab_im)

    labels_im = np.array(labels_im_list)
    
    return labels_im

In [4]:
# open df
df_path = os.path.join(path_save, f'df_{myWell}.pkl')
df = pd.read_pickle(df_path)

# Print the unique values in the nameRound column to verify myRound is present
print("Unique values in 'nameRound':", df['nameRound'].unique())

# Convert myRound to the correct string format
myRound_str = f'R{myRound}'

# Verify that myRound is now in the correct format
print("Formatted 'myRound':", myRound_str)

# Check which alignRound for the selected round
filtered_align_round = df.loc[df.nameRound == myRound_str, 'alignRound']
print("Filtered 'alignRound' for 'myRound_str':", filtered_align_round)

align_round_list = filtered_align_round.tolist()

if align_round_list:
    alignRound = align_round_list[0]
    print(f"Selected alignRound: {alignRound}")
else:
    raise ValueError(f"No alignRound found for myRound = {myRound_str}. Please check the input data.")

# Check which alignRound for the anchor round
anchor_round_str = f'R{anchor_round_selected}'
anchor_round_list = df.loc[df.nameRound == anchor_round_str, 'alignRound'].tolist()

if anchor_round_list:
    anchor_round = anchor_round_list[0]
    print(f"Selected anchor_round: {anchor_round}")
else:
    raise ValueError(f"No alignRound found for anchor_round_selected = {anchor_round_str}. Please check the input data.")

# Open labels and the control
labels_im = read_labels(path_labels, myWell, alignRound, anchor_round)

# Open the list of transformations for a given well
tmat_path = os.path.join(path_save, f'tmat_{myWell}.pkl')
with open(tmat_path, 'rb') as tmat_file:
    tmat = pickle.load(tmat_file)

# Select the appropriate transformation
t_org = tmat[int(alignRound)]


Unique values in 'nameRound': ['R0' 'R1' 'R2' 'R3' 'R4' 'R5']
Formatted 'myRound': R0
Filtered 'alignRound' for 'myRound_str': 0    0.0
1    0.0
2    0.0
3    0.0
Name: alignRound, dtype: float64
Selected alignRound: 0.0
Selected anchor_round: 1.0


## Use Affinder in Napari

Remember to set parameters of the plugin correctly:
- reference image
- image to transform
- 'similarity' as transformation

Information about Affinder can be found:
- https://github.com/jni/affinder
- https://www.napari-hub.org/plugins/affinder?imageModality=Fluorescence+microscopy

In [5]:
viewer = napari.Viewer()
l0 = viewer.add_image(labels_im[0], colormap='bop blue', blending='additive',name='ref')
l1 = viewer.add_image(labels_im[1], colormap='bop purple', blending='additive',name='moving')

qtwidget, widget = viewer.window.add_plugin_dock_widget(
        'affinder', 'Start affinder'
        )
widget.reference.bind(l0)
widget.moving.bind(l1)
widget()

## Collect transformation matrix

In [ ]:
mat = np.asarray(viewer.layers['moving'].affine)

## Visualize transformation from the collected matrix (optional)

In [ ]:
# optionally - test matrix
tfd_ndi = ndi.affine_transform(labels_im[1], np.linalg.inv(mat))
viewer.add_image(tfd_ndi, colormap='bop orange', blending='additive')

## Save transformation matrix

In [ ]:
def matrix_rc2xy(affine_matrix):
    swapped_cols = affine_matrix[:, [1, 0, 2]]
    swapped_rows = swapped_cols[[1, 0, 2], :]
    return swapped_rows


tmat[int(alignRound),:,:] = np.linalg.inv(matrix_rc2xy(mat))
tmat.dump(tmat_path)